In [1]:
import os 
import sys
import json
from pprint import pprint
from pathlib import Path

from IPython.display import display, HTML


In [2]:
def load_json(file_path):
    with open(file_path, "r") as file:
        dct = json.load(file)
    return dct


In [3]:
REPO_ROOT = Path.cwd().parent
RESULT_FILE_PATH = f"{REPO_ROOT}/outputs/harbor-swebench-2026-01-04::18-51-44/result.json"
TRAJECTORY_FILE_PATH = f"{REPO_ROOT}/outputs/harbor-swebench-2026-01-04::18-51-44/trajectory.json"


In [4]:
result = load_json(RESULT_FILE_PATH)
print("Total examples:", result["total"])
print("Accuracy:", result["accuracy"])


Total examples: 10
Accuracy: 0.5


In [5]:
def pretty_print_trajectory(messages):
    """
    Pretty print trajectory messages from Harbor format.
    Harbor messages have: step_id, timestamp, source, message, reasoning_content (optional), tool_calls (optional), observation (optional)
    """
    html_output = "<div style='font-family: Arial, sans-serif;'>"
    
    for i, turn in enumerate(messages):
        source = turn.get("source", "unknown")
        message = turn.get("message", "")
        reasoning_content = turn.get("reasoning_content", "")
        tool_calls = turn.get("tool_calls", [])
        observation = turn.get("observation", {})
        step_id = turn.get("step_id", i + 1)
        
        # Handle different content types
        if isinstance(message, dict):
            message = json.dumps(message, indent=2)
        elif isinstance(message, list):
            message = json.dumps(message, indent=2)
        
        # Escape HTML in content
        message = str(message).replace("<", "&lt;").replace(">", "&gt;")
        if reasoning_content:
            reasoning_content = str(reasoning_content).replace("<", "&lt;").replace(">", "&gt;")
        
        # Map source to role for styling
        if source == "user":
            role, color, bg_color, emoji = "user", "#1976D2", "#E3F2FD", "👤"
        elif source == "agent":
            role, color, bg_color, emoji = "assistant", "#7B1FA2", "#F3E5F5", "🤖"
        elif source == "system":
            role, color, bg_color, emoji = "system", "#666", "#f5f5f5", "⚙️"
        elif source == "tool" or source == "function":
            role, color, bg_color, emoji = "tool", "#F57C00", "#FFF3E0", "🔧"
        elif source == "error":
            role, color, bg_color, emoji = "error", "#D32F2F", "#FFEBEE", "⚠️"
        else:
            role, color, bg_color, emoji = source, "#757575", "#FAFAFA", "💬"
        
        html_output += f"""
        <div style="background-color: {bg_color}; 
                    border-left: 5px solid {color}; 
                    padding: 15px; 
                    margin: 12px 0; 
                    border-radius: 8px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <div style="display: flex; align-items: center; margin-bottom: 8px;">
                <span style="font-size: 1.2em; margin-right: 8px;">{emoji}</span>
                <strong style="color: {color}; font-size: 1.1em;">{role.upper()}</strong>
                <span style="color: #666; font-size: 0.9em; margin-left: 10px;">Step {step_id}</span>
            </div>
            <pre style="margin: 0; 
                       white-space: pre-wrap; 
                       word-wrap: break-word;
                       font-family: 'Courier New', monospace; 
                       background-color: white; 
                       padding: 10px; 
                       border-radius: 4px;
                       font-size: 0.95em;
                       line-height: 1.5;
                       color: #000;
                       overflow-x: auto;">{message}</pre>
        """
        
        # Add reasoning content if present
        if reasoning_content:
            html_output += f"""
            <div style="margin-top: 10px; padding: 10px; background-color: #FFF9C4; border-radius: 4px; border-left: 3px solid #FBC02D;">
                <strong style="color: #F57F17;">💭 Reasoning:</strong>
                <pre style="margin: 5px 0 0 0; white-space: pre-wrap; word-wrap: break-word; font-family: 'Courier New', monospace; font-size: 0.9em; color: #000;">{reasoning_content}</pre>
            </div>
            """
        
        # Add tool calls if present - format without tool_call_id
        if tool_calls:
            html_output += f"""
            <div style="margin-top: 10px; padding: 10px; background-color: #FFF3E0; border-radius: 4px; border-left: 3px solid #F57C00;">
                <strong style="color: #E65100;">🔧 Tool Calls:</strong>
            """
            for idx, tool_call in enumerate(tool_calls):
                # Remove tool_call_id for cleaner display
                function_name = tool_call.get("function_name", "unknown")
                arguments = tool_call.get("arguments", {})
                
                # Format arguments nicely
                args_str = json.dumps(arguments, indent=2).replace("<", "&lt;").replace(">", "&gt;")
                
                html_output += f"""
                <div style="margin-top: {'10px' if idx > 0 else '5px'}; padding: 8px; background-color: white; border-radius: 4px; border: 1px solid #FFB74D;">
                    <div style="font-weight: bold; color: #E65100; margin-bottom: 5px;">{function_name}()</div>
                    <pre style="margin: 0; white-space: pre-wrap; word-wrap: break-word; font-family: 'Courier New', monospace; font-size: 0.9em; color: #000;">{args_str}</pre>
                </div>
                """
            html_output += "</div>"
        
        # Add observation if present - format without source_call_id
        if observation:
            results = observation.get("results", [])
            if results:
                html_output += f"""
                <div style="margin-top: 10px; padding: 10px; background-color: #E8F5E9; border-radius: 4px; border-left: 3px solid #4CAF50;">
                    <strong style="color: #2E7D32;">👁️ Observation:</strong>
                """
                for idx, result in enumerate(results):
                    # Remove source_call_id for cleaner display
                    content = result.get("content", "")
                    # Escape HTML
                    content = str(content).replace("<", "&lt;").replace(">", "&gt;")
                    
                    html_output += f"""
                    <div style="margin-top: {'10px' if idx > 0 else '5px'}; padding: 8px; background-color: white; border-radius: 4px; border: 1px solid #81C784;">
                        <pre style="margin: 0; white-space: pre-wrap; word-wrap: break-word; font-family: 'Courier New', monospace; font-size: 0.9em; color: #000;">{content}</pre>
                    </div>
                    """
                html_output += "</div>"
            else:
                # Fallback for other observation formats
                obs_str = json.dumps(observation, indent=2).replace("<", "&lt;").replace(">", "&gt;")
                html_output += f"""
                <div style="margin-top: 10px; padding: 10px; background-color: #E8F5E9; border-radius: 4px; border-left: 3px solid #4CAF50;">
                    <strong style="color: #2E7D32;">👁️ Observation:</strong>
                    <pre style="margin: 5px 0 0 0; white-space: pre-wrap; word-wrap: break-word; font-family: 'Courier New', monospace; font-size: 0.9em; color: #000;">{obs_str}</pre>
                </div>
                """
        
        html_output += "</div>"
    
    html_output += "</div>"
    display(HTML(html_output))


In [6]:
trajectory = load_json(TRAJECTORY_FILE_PATH)


In [ ]:
# Display trajectory for the first example (base variant)
base_entry = next((entry for entry in trajectory['entries'] if entry['example_id'] == 0 and entry['variant'] == 'base'), None)
if base_entry:
    print(f"Example ID: {base_entry['example_id']}")
    print(f"Variant: {base_entry['variant']}")
    print(f"Output: {base_entry['output']}")
    print(f"Status: {base_entry['status']}")
    print(f"\nNumber of messages: {len(base_entry['messages'])}")
    pretty_print_trajectory(base_entry['messages'][:10])


SyntaxError: invalid syntax (3317883955.py, line 9)